**Performance Comparison with State-of-the-art**

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tabulate import tabulate
from tqdm import tqdm
import matplotlib.pyplot as plt
import torch.nn.functional as F

# ==========================================
# 1. DATA PREPROCESSING & UTILS
# ==========================================

def load_and_preprocess_data(filepath):
    """Loads dataset, encodes labels, and standardizes features."""
    data = pd.read_csv(filepath)
    X = data.iloc[:, :-1].values
    y = data.iloc[:, -1].values

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)
    class_names = label_encoder.classes_
    num_classes = len(class_names)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    return X_scaled, y_encoded, class_names, num_classes

def split_meta_data(y_encoded, num_classes, test_size=0.3, random_state=42):
    """Splits classes into meta-train and meta-test sets (disjoint classes)."""
    np.random.seed(random_state)
    all_classes = np.unique(y_encoded)
    meta_train_classes = np.random.choice(
        all_classes, int((1 - test_size) * num_classes), replace=False)
    meta_test_classes = np.array([c for c in all_classes if c not in meta_train_classes])

    train_mask = np.isin(y_encoded, meta_train_classes)
    test_mask = np.isin(y_encoded, meta_test_classes)

    return train_mask, test_mask

def sample_episode(X, y, n_way, k_shot, q_query):
    """Samples an N-way K-shot episode."""
    unique_classes = np.unique(y)
    n_way = min(n_way, len(unique_classes))
    selected_classes = np.random.choice(unique_classes, n_way, replace=False)

    support, support_labels = [], []
    query, query_labels = [], []

    for idx, cls in enumerate(selected_classes):
        cls_indices = np.where(y == cls)[0]
        # Ensure enough samples
        if len(cls_indices) < k_shot + q_query:
            replace = True
        else:
            replace = False

        # Sample support
        support_indices = np.random.choice(cls_indices, k_shot, replace=replace)
        support.append(X[support_indices])
        support_labels.extend([idx] * k_shot)

        # Sample query
        remaining_indices = np.setdiff1d(cls_indices, support_indices)
        if len(remaining_indices) < q_query:
             query_indices = np.random.choice(cls_indices, q_query, replace=True)
        else:
             query_indices = np.random.choice(remaining_indices, q_query, replace=False)

        query.append(X[query_indices])
        query_labels.extend([idx] * q_query)

    support = torch.FloatTensor(np.vstack(support))
    query = torch.FloatTensor(np.vstack(query))
    support_labels = torch.LongTensor(support_labels)
    query_labels = torch.LongTensor(query_labels)

    return support, support_labels, query, query_labels

# ==========================================
# 2. BASELINE ARCHITECTURES (REFS 12-15)
# ==========================================

class Yan_TransformerBackbone(nn.Module):
    """
    Reproduction of Ref [12]: Transformer-based architecture.
    Uses a linear projection followed by a Transformer Encoder Layer.
    """
    def __init__(self, input_dim, embed_dim=128):
        super().__init__()
        self.project = nn.Linear(input_dim, 128)
        self.encoder_layer = nn.TransformerEncoderLayer(d_model=128, nhead=4, dim_feedforward=256, batch_first=True)
        self.transformer = nn.TransformerEncoder(self.encoder_layer, num_layers=2)
        self.fc_out = nn.Linear(128, embed_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        # x: [batch, features] -> [batch, 1, features] for transformer
        x = self.relu(self.project(x))
        x = x.unsqueeze(1)
        x = self.transformer(x)
        x = x.squeeze(1)
        return self.fc_out(x)

class Farzaneh_InceptionBackbone(nn.Module):
    """
    Reproduction of Ref [13]: 1D-Inception architecture.
    Combines 1x1, 3x3, 5x5 kernels.
    """
    def __init__(self, input_dim, embed_dim=128):
        super().__init__()
        self.input_dim = input_dim
        # Branch 1: 1x1 conv
        self.branch1 = nn.Conv1d(1, 32, kernel_size=1)
        # Branch 2: 1x1 conv -> 3x3 conv
        self.branch2 = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=1),
            nn.ReLU(),
            nn.Conv1d(32, 32, kernel_size=3, padding=1)
        )
        # Branch 3: 1x1 conv -> 5x5 conv
        self.branch3 = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=1),
            nn.ReLU(),
            nn.Conv1d(32, 32, kernel_size=5, padding=2)
        )
        # Branch 4: MaxPool -> 1x1 conv
        self.branch4 = nn.Sequential(
            nn.MaxPool1d(kernel_size=3, stride=1, padding=1),
            nn.Conv1d(1, 32, kernel_size=1)
        )

        # Output calculation: 32*4 channels * input_dim (since global pool not applied yet)
        # We use Global Avg Pooling after concatenation
        self.fc = nn.Linear(32 * 4, embed_dim)

    def forward(self, x):
        # x: [batch, features] -> [batch, 1, features]
        x = x.unsqueeze(1)
        b1 = self.branch1(x)
        b2 = self.branch2(x)
        b3 = self.branch3(x)
        b4 = self.branch4(x)

        out = torch.cat([b1, b2, b3, b4], dim=1)
        # Global Average Pooling
        out = F.adaptive_avg_pool1d(out, 1).squeeze(2)
        return self.fc(out)

class Islam_VAEEncoderBackbone(nn.Module):
    """
    Reproduction of Ref [14]: VAE Encoder architecture.
    Uses a standard MLP structure typically found in VAE encoders.
    """
    def __init__(self, input_dim, embed_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, embed_dim) # Latent mean z_mu
        )

    def forward(self, x):
        return self.net(x)

class Farzaneh_ResNetBackbone(nn.Module):
    """
    Reproduction of Ref [15]: ResNet-1D architecture.
    """
    def __init__(self, input_dim, embed_dim=128):
        super().__init__()
        self.in_planes = 64
        self.conv1 = nn.Conv1d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(64)

        # Residual Block
        self.layer1 = self._make_layer(64, stride=1)
        self.layer2 = self._make_layer(128, stride=2)

        self.fc = nn.Linear(128, embed_dim)

    def _make_layer(self, planes, stride):
        layers = []
        layers.append(nn.Sequential(
            nn.Conv1d(self.in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm1d(planes),
            nn.ReLU()
        ))
        # Shortcut logic simplified for 1D
        self.in_planes = planes
        return nn.Sequential(*layers)

    def forward(self, x):
        x = x.unsqueeze(1)
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = F.adaptive_avg_pool1d(out, 1).squeeze(2)
        return self.fc(out)

# ==========================================
# 3. GENERIC PROTONET (For Baselines)
# ==========================================

class GenericProtoNet(nn.Module):
    """
    Generic Prototypical Network Wrapper for Baseline Architectures.
    Does NOT use MSPL or Adversarial Consistency Loss.
    """
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, x):
        return self.backbone(x)

    def compute_loss(self, support, support_labels, query, query_labels):
        support_emb = self(support)
        query_emb = self(query)

        prototypes = []
        unique_labels = torch.unique(support_labels)
        for label in unique_labels:
            prototypes.append(support_emb[support_labels == label].mean(dim=0))
        prototypes = torch.stack(prototypes)

        # Map labels to 0..N-1
        label_map = {lbl.item(): i for i, lbl in enumerate(unique_labels)}
        query_labels_mapped = torch.LongTensor([label_map[lbl.item()] for lbl in query_labels]).to(query.device)

        dists = torch.cdist(query_emb, prototypes)
        return nn.CrossEntropyLoss()(-dists, query_labels_mapped)

# ==========================================
# 4. PROPOSED MSPL-IDS ARCHITECTURE
# ==========================================

class MSPLNet(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, embed_dim=128, num_spaces=3):
        super().__init__()
        self.num_spaces = num_spaces
        self.embed_dim = embed_dim

        # Shared extractor
        self.shared_net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.BatchNorm1d(hidden_dim)
        )

        # Multiple heads
        self.projection_heads = nn.ModuleList([
            nn.Sequential(nn.Linear(hidden_dim, embed_dim), nn.ReLU())
            for _ in range(num_spaces)
        ])

        # Learnable attention
        self.space_weights = nn.Parameter(torch.ones(num_spaces) / num_spaces)
        self.adv_coeff = 0.5

    def forward(self, x, return_all=False):
        shared = self.shared_net(x)
        projections = [head(shared) for head in self.projection_heads]

        if return_all:
            return projections

        weights = torch.softmax(self.space_weights, dim=0)
        combined = sum(w * p for w, p in zip(weights, projections))
        return combined

    def adversarial_consistency_loss(self, clean_x, adv_x, y):
        clean_embs = self(clean_x, return_all=True)
        adv_embs = self(adv_x, return_all=True)
        loss = 0
        for clean, adv in zip(clean_embs, adv_embs):
            # Compute prototypes from clean support
            prototypes = []
            for cls in torch.unique(y):
                prototypes.append(clean[y==cls].mean(dim=0))
            prototypes = torch.stack(prototypes)

            clean_logits = -torch.cdist(clean, prototypes)
            adv_logits = -torch.cdist(adv, prototypes)

            clean_probs = F.softmax(clean_logits, dim=1)
            adv_probs = F.log_softmax(adv_logits, dim=1)

            loss += nn.KLDivLoss(reduction='batchmean')(adv_probs, clean_probs)
        return loss / self.num_spaces

# ==========================================
# 5. ATTACK GENERATORS (FGSM & PGD)
# ==========================================

def generate_attack(model, x, y, method='FGSM', epsilon=0.1, alpha=0.01, steps=10):
    """Generates adversarial examples for any ProtoNet-like model."""
    x_adv = x.clone().detach().requires_grad_(True)

    if method == 'FGSM':
        # Single step
        if isinstance(model, MSPLNet):
            embs = model(x_adv, return_all=True)[0] # Use first space for grad
        else:
            embs = model(x_adv)

        prototypes = []
        for cls in torch.unique(y):
            prototypes.append(embs[y==cls].mean(dim=0))
        prototypes = torch.stack(prototypes)

        # Fix label mapping for grad calculation
        label_map = {lbl.item(): i for i, lbl in enumerate(torch.unique(y))}
        y_mapped = torch.LongTensor([label_map[lbl.item()] for lbl in y]).to(x.device)

        loss = nn.CrossEntropyLoss()(-torch.cdist(embs, prototypes), y_mapped)
        model.zero_grad()
        loss.backward()
        x_adv = x + epsilon * x_adv.grad.sign()

    elif method == 'PGD':
        # Iterative
        for _ in range(steps):
            x_adv.requires_grad = True
            if isinstance(model, MSPLNet):
                embs = model(x_adv, return_all=True)[0]
            else:
                embs = model(x_adv)

            prototypes = []
            for cls in torch.unique(y):
                prototypes.append(embs[y==cls].mean(dim=0))
            prototypes = torch.stack(prototypes)

            label_map = {lbl.item(): i for i, lbl in enumerate(torch.unique(y))}
            y_mapped = torch.LongTensor([label_map[lbl.item()] for lbl in y]).to(x.device)

            loss = nn.CrossEntropyLoss()(-torch.cdist(embs, prototypes), y_mapped)
            model.zero_grad()
            loss.backward()

            adv_x = x_adv + alpha * x_adv.grad.sign()
            eta = torch.clamp(adv_x - x, min=-epsilon, max=epsilon)
            x_adv = (x + eta).detach()

    return x_adv.detach()

# ==========================================
# 6. TRAINING & EVALUATION LOOPS
# ==========================================

def train_model(model, X, y, n_way, k_shot, q_query, epochs=5, episodes=50,
               is_mspl_robust=False, lr=0.001):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    model.train()

    for epoch in range(epochs):
        avg_loss = 0
        for _ in range(episodes):
            support, s_y, query, q_y = sample_episode(X, y, n_way, k_shot, q_query)

            optimizer.zero_grad()

            if isinstance(model, MSPLNet):
                # MSPL Logic
                loss = 0
                projections_s = model(support, return_all=True)
                projections_q = model(query, return_all=True)

                # Proto Loss per space
                for ps, pq in zip(projections_s, projections_q):
                    protos = []
                    unique = torch.unique(s_y)
                    for u in unique:
                        protos.append(ps[s_y==u].mean(dim=0))
                    protos = torch.stack(protos)

                    label_map = {u.item(): i for i, u in enumerate(unique)}
                    q_y_mapped = torch.LongTensor([label_map[l.item()] for l in q_y])
                    loss += nn.CrossEntropyLoss()(-torch.cdist(pq, protos), q_y_mapped)

                loss /= model.num_spaces

                # Robustness Loss
                if is_mspl_robust:
                    adv_query = generate_attack(model, query, q_y, method='FGSM', epsilon=0.1)
                    loss += 0.5 * model.adversarial_consistency_loss(query, adv_query, q_y)

            else:
                # Standard Baseline Logic
                loss = model.compute_loss(support, s_y, query, q_y)

            loss.backward()
            optimizer.step()
            avg_loss += loss.item()

    return model

def evaluate(model, X, y, n_way, k_shot, q_query, episodes=50, attack_type=None):
    model.eval()
    accs = []

    for _ in range(episodes):
        support, s_y, query, q_y = sample_episode(X, y, n_way, k_shot, q_query)

        # Apply attack if needed
        if attack_type == 'FGSM':
            query = generate_attack(model, query, q_y, 'FGSM', epsilon=0.1)
        elif attack_type == 'PGD':
            query = generate_attack(model, query, q_y, 'PGD', epsilon=0.1, steps=7, alpha=0.02)

        with torch.no_grad():
            if isinstance(model, MSPLNet):
                # Ensemble inference
                projections_s = model(support, return_all=True)
                projections_q = model(query, return_all=True)
                weights = torch.softmax(model.space_weights, dim=0)

                combined_logits = 0
                unique = torch.unique(s_y)

                for idx, (ps, pq) in enumerate(zip(projections_s, projections_q)):
                    protos = []
                    for u in unique:
                        protos.append(ps[s_y==u].mean(dim=0))
                    protos = torch.stack(protos)
                    logits = -torch.cdist(pq, protos)
                    combined_logits += weights[idx] * logits

            else:
                # Standard inference
                s_emb = model(support)
                q_emb = model(query)
                protos = []
                unique = torch.unique(s_y)
                for u in unique:
                    protos.append(s_emb[s_y==u].mean(dim=0))
                protos = torch.stack(protos)
                combined_logits = -torch.cdist(q_emb, protos)

        # Accuracy
        label_map = {u.item(): i for i, u in enumerate(unique)}
        q_y_mapped = torch.LongTensor([label_map[l.item()] for l in q_y])
        preds = combined_logits.argmax(dim=1)
        accs.append((preds == q_y_mapped).float().mean().item())

    return np.mean(accs)

# ==========================================
# 7. MAIN EXPERIMENT RUNNER
# ==========================================

if __name__ == "__main__":
    # --- CONFIG ---
    FILEPATH = '/content/drive/MyDrive/ID2024-5G/Preprocessed_5G.csv'
    N_WAY = 5
    Q_QUERY = 15
    EPISODES_TEST = 50 # Increase for final paper results

    # Define Scenarios: (K_SHOT values to test)
    SCENARIOS = [1, 5, 10]

    print(">>> Loading Data...")
    try:
        X_scaled, y_encoded, class_names, num_classes = load_and_preprocess_data(FILEPATH)
    except FileNotFoundError:
        print("Error: CSV not found. Using random mock data for demonstration.")
        X_scaled = np.random.randn(1000, 78)
        y_encoded = np.random.randint(0, 10, 1000)
        num_classes = 10

    train_mask, test_mask = split_meta_data(y_encoded, num_classes)
    X_train, y_train = X_scaled[train_mask], y_encoded[train_mask]
    X_test, y_test = X_scaled[test_mask], y_encoded[test_mask]

    input_dim = X_train.shape[1]

    # Loop over different K-shot scenarios
    for k_shot in SCENARIOS:
        print(f"\n{'='*60}")
        print(f"STARTING SCENARIO: {N_WAY}-Way {k_shot}-Shot")
        print(f"{'='*60}")

        # Re-initialize models for each scenario to ensure fair training from scratch
        models_config = [
            ("Ref [12] (Yan et al.)", GenericProtoNet(Yan_TransformerBackbone(input_dim))),
            ("Ref [13] (Farzaneh et al.)", GenericProtoNet(Farzaneh_InceptionBackbone(input_dim))),
            ("Ref [14] (Islam et al.)", GenericProtoNet(Islam_VAEEncoderBackbone(input_dim))),
            ("Ref [15] (Farzaneh et al.)", GenericProtoNet(Farzaneh_ResNetBackbone(input_dim))),
            ("Ours (Standard MSPL)", MSPLNet(input_dim)),
            ("Ours (Robust MSPL-IDS)", MSPLNet(input_dim)) # Will be trained robustly
        ]

        results = []

        for name, model in models_config:
            print(f"\nTraining {name} ({k_shot}-shot)...")

            is_robust = "Robust" in name
            train_model(model, X_train, y_train, N_WAY, k_shot, Q_QUERY,
                        epochs=5, is_mspl_robust=is_robust)

            print(f"Evaluating {name}...")
            acc_clean = evaluate(model, X_test, y_test, N_WAY, k_shot, Q_QUERY)
            acc_fgsm = evaluate(model, X_test, y_test, N_WAY, k_shot, Q_QUERY, attack_type='FGSM')
            acc_pgd = evaluate(model, X_test, y_test, N_WAY, k_shot, Q_QUERY, attack_type='PGD')

            results.append({
                "Method": name,
                "Clean Acc": f"{acc_clean:.4f}",
                "FGSM Acc": f"{acc_fgsm:.4f}",
                "PGD Acc": f"{acc_pgd:.4f}"
            })

        print(f"\n>>> RESULTS TABLE: {N_WAY}-Way {k_shot}-Shot")
        print(tabulate(results, headers="keys", tablefmt="grid"))



>>> Loading Data...

STARTING SCENARIO: 5-Way 1-Shot

Training Ref [12] (Yan et al.) (1-shot)...
Evaluating Ref [12] (Yan et al.)...

Training Ref [13] (Farzaneh et al.) (1-shot)...
Evaluating Ref [13] (Farzaneh et al.)...

Training Ref [14] (Islam et al.) (1-shot)...
Evaluating Ref [14] (Islam et al.)...

Training Ref [15] (Farzaneh et al.) (1-shot)...
Evaluating Ref [15] (Farzaneh et al.)...

Training Ours (Standard MSPL) (1-shot)...
Evaluating Ours (Standard MSPL)...

Training Ours (Robust MSPL-IDS) (1-shot)...
Evaluating Ours (Robust MSPL-IDS)...

>>> RESULTS TABLE: 5-Way 1-Shot
+----------------------------+-------------+------------+-----------+
| Method                     |   Clean Acc |   FGSM Acc |   PGD Acc |
+============================+=============+============+===========+
| Ref [12] (Yan et al.)      |      0.9262 |     0.6449 |    0.8471 |
+----------------------------+-------------+------------+-----------+
| Ref [13] (Farzaneh et al.) |      0.4933 |     0.2756 |   